In [4]:
import pandas as pd
import os
from sklearn.model_selection import train_test_split

df = pd.read_csv("data/raw/complaints_raw.csv")
counts = df["issue"].value_counts()
df = df[df["issue"].isin(counts[counts >= 1000].index)].copy()

df["narrative"] = (df["narrative"]
    .str.replace(r"XX/XX/\S+", " ", regex=True)   # hidden dates
    .str.replace(r"X{2,}", " ", regex=True)       # hidden names / numbers
    .str.replace(r"[{}]", "", regex=True)         # {$0.00} -> $0.00
    .str.replace(r"\s+", " ", regex=True)         # squash extra spaces
    .str.strip())
df["words"] = df["narrative"].str.split().str.len()
df = df[df["words"] >= 10].copy()
print(df.shape)
print(df["words"].describe().round(0))
print("\n--- Same example, cleaned ---\n")
print(df["narrative"].iloc[0][:600])

(102402, 13)
count    102402.0
mean        203.0
std         218.0
min          10.0
25%          69.0
50%         153.0
75%         261.0
max        6263.0
Name: words, dtype: float64

--- Same example, cleaned ---

Credit card issuer : Imprint ( and , formerly ) I am filing this complaint due to an improper interest charge and lack of resolution following the migration of my credit card account from to Imprint. I originally had a credit card that I only used a few times. My most recent statement was dated showing a $0.00 balance. I had paid my previous statement in full and on time. Therefore, I had no outstanding balance or trailing interest at the time of migration. notified customers that the credit card program would migrate from to Imprint on and that balances, purchases, and credits would transfer 


In [5]:
train, test = train_test_split(df, test_size=1000, stratify=df["issue"], random_state=42)
gold = test.sample(100, random_state=42)

os.makedirs("data/clean", exist_ok=True)
train.to_csv("data/clean/train.csv", index=False)
test.to_csv("data/clean/test_1000.csv", index=False)
gold.to_csv("data/clean/gold_100.csv", index=False)

print("Train:", train.shape, "| Test:", test.shape, "| Gold:", gold.shape)
print((test["issue"].value_counts(normalize=True) * 100).round(1))

Train: (101402, 13) | Test: (1000, 13) | Gold: (100, 13)
issue
Problem with a purchase shown on your statement                    26.4
Other features, terms, or problems                                 11.7
Getting a credit card                                              11.4
Fees or interest                                                   11.1
Problem with a company's investigation into an existing problem     8.7
Closing your account                                                7.6
Incorrect information on your report                                6.7
Problem when making payments                                        5.9
Advertising and marketing, including promotional offers             4.9
Trouble using your card                                             4.3
Struggling to pay your bill                                         1.3
Name: proportion, dtype: float64
